# LUMI 2단계: 동영상 내 광과민성 발작(PSE) 5대 위험 트리거 구간 판정 알고리즘 (`video_discriminator_1.ipynb`)

본 노트북은 `image_discriminator_2_adstripe.ipynb`의 단일 이미지 물리 휘도·색도·공간 주파수(`1.0 ~ 10.0 cpd`) 판정 알고리즘을 시계열 동영상 프레임 분석으로 확장하여, **5가지 위험 트리거별 위험 영상 구간(프레임·시간)과 판정 근거(임계치 달성 정도)**를 데이터프레임으로 산출합니다.

---

### 1. 5대 위험 트리거 판정 기준 요약

| 번호 | 위험 구분 | 단일 프레임 공간·물리 조건 | 영역 임계치 | 시계열(시간·점멸) 위험 구간 판정 조건 |
| :---: | :--- | :--- | :---: | :--- |
| **1** | **휘도 플래시** | 프레임 평균 휘도 대비 $+30\%$ 이상 & $\ge 20\text{ cd/m}^2$ 고휘도 | **25%** | `(위험, 안전)` 쌍이 **1초에 4번 이상($> 3\text{ Hz}$)** 발생 & 위험 프레임 간 안전 프레임 **9프레임 이하** |
| **2** | **적색 플래시** | 선형 RGB $R/(R+G+B) \ge 0.8$ & 색도차 $\Delta u'v' > 0.2$ | **25%** | `(위험, 안전)` 쌍이 **1초에 4번 이상($> 3\text{ Hz}$)** 발생 & 위험 프레임 간 안전 프레임 **9프레임 이하** |
| **3** | **지속 플래시** | 프레임 평균 휘도 대비 $+30\%$ 이상 & $\ge 20\text{ cd/m}^2$ 고휘도 | **40%** | **안전 프레임 $\rightarrow$ 위험 프레임으로 전환**된 후 **1.0초 이상 연속** 지속 |
| **4** | **줄무늬 패턴** | ITU-R 명암 대비 충족 & **$1.0 \sim 10.0\text{ cpd}$ 연속 직선 줄무늬 $\ge 5$쌍** | **25%** | 위험 프레임이 **0.5초 이상 연속 지속**되거나, `(위험, 안전)` 쌍이 **1초에 4번 이상(간격 $\le 9$프레임)** 점멸 |
| **5** | **고대비** | $L_{\text{LOW}} < 160$일 때 $\Delta L \ge 20\text{ cd/m}^2$, $L_{\text{LOW}} \ge 160$일 때 Michelson $> 1/17$ | **40%** | 위험 프레임이 **0.5초 이상 연속 지속**되거나, `(위험, 안전)` 쌍이 **1초에 4번 이상(간격 $\le 9$프레임)** 점멸 |

---

### 2. `줄무늬 패턴` 및 `고대비`의 지속 시간·점멸 기준 관련 문헌 출처
1. **줄무늬 패턴 0.5초 지속 및 25% 면적 기준**:
   - **폴더 내 논문**: `C:\LUMI\짜바리\Detection_of_Flashy_Segment_in_YouTube_Videos_using_Computer_Vision_Technique_to_Prevent_Seizure_in_Epilepsy_Patients.pdf` (Tripathi et al., 2023, 1쪽 Introduction)
   - **인용 원문**: **Wilkins, A., Emmett, J., & Harding, G. (2005)**, *"Characterising the images that precipitate seizures and optimizing guidelines to prevent them"*, *Epilepsia*, 46(8), 1212–1218 (및 영국 Ofcom 가이드라인) — 화면의 $1/4$(25%)을 초과하는 줄무늬 패턴이 **0.5초(`> 0.5 s`) 이상 지속**될 때 발작 유발 위험으로 규정.
2. **정지 패턴(40%) vs 동적·점멸 패턴(25%) 및 고대비 기준**:
   - **폴더 내 표준 문서**: `C:\LUMI\주요 논문\PSE 관련\ITU-R_BT.1702-3.pdf` (3쪽 Guideline 1, 6쪽 Attachment 1 to Annex 1)
   - **폴더 내 의학 논문**: `C:\LUMI\주요 논문\PSE 관련\1-s2.0-S0003687008001282-main.pdf` (Harding & Harding, 2010, 505~507쪽) 및 `C:\LUMI\짜바리\US8295359.pdf` (Auratechnic 특허, 프레임 내 공간적 고대비 제한).
3. **표준 시청거리, 시야각($2\theta$) 및 3Hz(초당 3회 플래시 한계 = 4쌍 이상 위험) 기준**:
   - **폴더 내 논문**: `C:\LUMI\주요 논문\PSE 관련\3694790.pdf` (Jordan & Vanderheiden, ACM TACCESS 2024, Table 2, Table 6, Table 7, Appendix A).

### [코드 셀 1-A] 비디오 메타데이터 확인 · 물리 휘도 변환 · 표준 시청거리 기반 시야각 및 `cpd` 기준 정립

In [1]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

# 분석할 기본 동영상 경로 및 디스플레이 설정
VIDEO_PATH = r"C:\LUMI\영상\PSE_triggering_dataset\1.mp4"
DEFAULT_DEVICE = "laptop"     # 'laptop' (56.0 cm) 또는 'desktop' (68.0 cm)
DEFAULT_DISPLAY_AREA = 15     # 디스플레이 대각선 크기 (inch)
MAX_SIDE = 720                # 동영상 프레임 분석 시 최대 해상도 변 길이 (속도 및 정확도 균형)

@torch.no_grad()
def srgb_to_linear_and_luminance(rgb_255: torch.Tensor):
    """방식 A (ITU-R BT.1702-3 / WCAG 2.2): sRGB [0,255] -> 선형 RGB [0,1] 및 물리 휘도 L [0, 200] cd/m^2"""
    
    c_s = torch.clamp(rgb_255 / 255.0, 0.0, 1.0) # sRGB Color
    """0~255를 0~1로 스케일링"""
    
    c_lin = torch.where(
        c_s <= 0.04045,
        c_s / 12.92,
        torch.pow((c_s + 0.055) / 1.055, 2.4)
    ) # Linear Color
    """
    사람의 눈은 밝은 곳보다 어두운 곳의 미세한 밝기 변화에 훨씬 민감함.
    제한된 숫자(0~255)에 어두운 색조를 촘촘하게 저장하기 위해 실제 빛의 에너지를 그대로 표현 x
    => 원래 sRGB는 감마 압축 (1/2.2)승 __ 여기선 2.4로 쓰임.
    픽셀값 0~10 수준에서는 미분값이 너무 낮음 -> 0.04045 이하에서는 직선으로 변환
    0.055, 1.055는 미분가능하게 만들기 위한 숫자
    """
    y_rel = 0.2126 * c_lin[..., 0] + 0.7152 * c_lin[..., 1] + 0.0722 * c_lin[..., 2] # 상대 휘도
    """ITU-R에서 정한 인간 시각 밝기 가중치"""
    lum_cd = 200.0 * y_rel # 물리 휘도
    """ITU-R에서 표준 일반 디스플레이(SDR)에서 최대 흰색(y_rel = 1.0)의 물리 휘도를 200cd/m^2로 기준 삼는다"""
    return c_lin, y_rel, lum_cd

def compute_viewing_angle_and_cpd_spec(
    img_w: int,
    img_h: int,
    device: str = "laptop",
    display_area: int = 15,
    cpd_min: float = 1.0,
    cpd_max: float = 10.0
) -> dict:
    """
    표준 시청거리 가정을 통해 디스플레이 시야각(Visual Angle)과 가로·세로 1도당 픽셀 수,
    그리고 위험 공간 주파수(1.0 ~ 10.0 cpd) 판정을 위한 기준을 정립합니다.
    - device: 'laptop' (표준 시청거리 56.0 cm) 또는 'desktop' (표준 시청거리 68.0 cm)
    - display_area: 디스플레이 대각선 크기 (inch, 기본값 15)
    """
    if device == "laptop":
        viewing_dist_cm = 56.0
    elif device == "desktop":
        viewing_dist_cm = 68.0
    else:
        raise ValueError(f"device 인자는 'laptop' 또는 'desktop'만 허용됩니다. (입력값: {device!r})")

    if int(display_area) <= 0:
        raise ValueError(f"display_area(디스플레이 크기 inch)는 양의 정수여야 합니다. (입력값: {display_area})")

    # 1. 16:9 표준 화면비 기준 물리적 가로/세로 길이(cm) 계산 (1 inch = 2.54 cm)
    diag_cm = float(int(display_area)) * 2.54
    aspect_diag = math.sqrt(16.0 ** 2 + 9.0 ** 2)
    screen_w_cm = diag_cm * (16.0 / aspect_diag)
    screen_h_cm = diag_cm * (9.0 / aspect_diag)

    # 2. 시야각 계산: 2*theta = 2 * arctan(x / (2 * d)) (3694790.pdf Appendix A)
    fov_w_deg = 2.0 * math.degrees(math.atan(screen_w_cm / (2.0 * viewing_dist_cm)))
    fov_h_deg = 2.0 * math.degrees(math.atan(screen_h_cm / (2.0 * viewing_dist_cm)))

    # 3. 가로·세로 시야각 1도당 픽셀 수
    px_per_deg_w = float(img_w) / fov_w_deg
    px_per_deg_h = float(img_h) / fov_h_deg
    px_per_deg = 0.5 * (px_per_deg_w + px_per_deg_h)

    return {
        "device": device,
        "display_area_inch": int(display_area),
        "viewing_dist_cm": viewing_dist_cm,
        "screen_w_cm": screen_w_cm,
        "screen_h_cm": screen_h_cm,
        "fov_w_deg": fov_w_deg,
        "fov_h_deg": fov_h_deg,
        "px_per_deg_w": px_per_deg_w,
        "px_per_deg_h": px_per_deg_h,
        "px_per_deg": px_per_deg,
        "cpd_min": cpd_min,
        "cpd_max": cpd_max,
    }

def get_video_metadata(video_path: str, max_side: int = MAX_SIDE) -> dict:
    """동영상 파일의 원본/분석 해상도, FPS, 총 프레임 수, 재생 시간을 확인합니다."""
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"동영상 파일을 찾을 수 없습니다: {video_path}")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"동영상을 열 수 없습니다: {video_path}")
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 0 or math.isnan(fps):
        fps = 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if max(orig_h, orig_w) > max_side:
        scale = max_side / float(max(orig_h, orig_w))
        proc_w, proc_h = int(orig_w * scale), int(orig_h * scale)
    else:
        proc_w, proc_h = orig_w, orig_h

    return {
        "video_path": video_path,
        "orig_w": orig_w,
        "orig_h": orig_h,
        "proc_w": proc_w,
        "proc_h": proc_h,
        "fps": fps,
        "total_frames": total_frames,
        "duration_sec": total_frames / fps if total_frames > 0 else 0.0
    }

# 메타데이터 및 CPD 측정 기준 설정값 확인
_meta = get_video_metadata(VIDEO_PATH, max_side=MAX_SIDE)
_spec = compute_viewing_angle_and_cpd_spec(_meta["proc_w"], _meta["proc_h"], device=DEFAULT_DEVICE, display_area=DEFAULT_DISPLAY_AREA)
df_video_spec = pd.DataFrame([{
    "분석 해상도 (가로×세로)": f"{_meta['proc_w']} × {_meta['proc_h']}",
    "프레임 레이트 (FPS)": f"{_meta['fps']:.2f} fps",
    "총 프레임 수 (재생 시간)": f"{_meta['total_frames']} 프레임 ({_meta['duration_sec']:.2f}s)",
    "기기 및 화면 크기": f"{_spec['device']} ({_spec['display_area_inch']} inch, 시청거리 {_spec['viewing_dist_cm']:.0f}cm)",
    "디스플레이 시야각": f"{_spec['fov_w_deg']:.2f}° × {_spec['fov_h_deg']:.2f}°",
    "1°당 픽셀 수 (가로/세로)": f"{_spec['px_per_deg_w']:.2f} / {_spec['px_per_deg_h']:.2f} px/deg",
    "위험 공간주파수 범위": f"{_spec['cpd_min']:.1f} ~ {_spec['cpd_max']:.1f} cpd"
}], index=["입력 영상 및 CPD 설정 정보"])
display(df_video_spec)

,분석 해상도 (가로×세로),프레임 레이트 (FPS),총 프레임 수 (재생 시간),기기 및 화면 크기,디스플레이 시야각,1°당 픽셀 수 (가로/세로),위험 공간주파수 범위
입력 영상 및 CPD 설정 정보,720 × 405,29.97 fps,518 프레임 (17.28s),"laptop (15 inch, 시청거리 56cm)",33.03° × 18.94°,21.80 / 21.39 px/deg,1.0 ~ 10.0 cpd


### [코드 셀 1-B] 단일 프레임 내 5대 위험 트리거 지표 및 방식 1 확률(임계치 달성도) 동시 평가 함수

In [2]:
def count_opposing_stripe_pairs_1d_cpd(
    profile: np.ndarray,
    px_per_deg: float,
    cpd_min: float = 1.0,
    cpd_max: float = 10.0
) -> tuple:
    """
    1차원 휘도 프로파일(1 px 간격 샘플링)에서:
    1) ITU-R BT.1702-3 명암 대비 기준(>=20 cd/m^2 또는 Michelson > 1/17)을 만족하고,
    2) 인접한 봉우리-골짜기 간격이 공간 주파수 1.0 ~ 10.0 cpd 범위 안에 들어오는
       '연속된(끊기지 않은) 줄무늬 쌍(contiguous light-dark pairs)'의 최대 개수와 평균 cpd를 반환합니다.
    """
    if profile.size < 4:
        return 0, 0.0

    extrema_indices = [0]
    extrema_vals = [float(profile[0])]
    direction = 0

    for idx in range(1, profile.size):
        v = float(profile[idx])
        diff = v - extrema_vals[-1]
        if direction >= 0 and diff > 0:
            if direction == 0:
                extrema_vals.append(v)
                extrema_indices.append(idx)
            else:
                extrema_vals[-1] = v
                extrema_indices[-1] = idx
            direction = 1
        elif direction <= 0 and diff < 0:
            if direction == 0:
                extrema_vals.append(v)
                extrema_indices.append(idx)
            else:
                extrema_vals[-1] = v
                extrema_indices[-1] = idx
            direction = -1
        elif direction == 1 and (extrema_vals[-1] - v) >= 5.0:
            extrema_vals.append(v)
            extrema_indices.append(idx)
            direction = -1
        elif direction == -1 and (v - extrema_vals[-1]) >= 5.0:
            extrema_vals.append(v)
            extrema_indices.append(idx)
            direction = 1

    max_contiguous_transitions = 0
    best_run_cpds = []
    cur_transitions = 0
    cur_cpds = []

    for i in range(len(extrema_vals) - 1):
        l_high = max(extrema_vals[i], extrema_vals[i + 1])
        l_low = min(extrema_vals[i], extrema_vals[i + 1])
        cond_low = (l_low < 160.0) and ((l_high - l_low) >= 20.0)
        cond_high = (l_low >= 160.0) and (((l_high - l_low) / max(l_high + l_low, 1e-6)) > (1.0 / 17.0))
        contrast_ok = cond_low or cond_high

        half_cycle_px = float(extrema_indices[i + 1] - extrema_indices[i])
        full_cycle_px = 2.0 * max(half_cycle_px, 1e-6)
        local_cpd = px_per_deg / full_cycle_px
        cpd_ok = (cpd_min <= local_cpd <= cpd_max)

        if contrast_ok and cpd_ok:
            cur_transitions += 1
            cur_cpds.append(local_cpd)
            if cur_transitions > max_contiguous_transitions:
                max_contiguous_transitions = cur_transitions
                best_run_cpds = list(cur_cpds)
        else:
            cur_transitions = 0
            cur_cpds = []

    max_pairs = max_contiguous_transitions // 2
    mean_cpd = float(np.mean(best_run_cpds)) if (max_pairs > 0 and len(best_run_cpds) > 0) else 0.0
    return max_pairs, mean_cpd

def precompute_stripe_scan_lines(w: int, h: int, cpd_spec: dict) -> list:
    """동영상의 모든 프레임에서 동일한 스캔 좌표를 재사용할 수 있도록 60개 직선 좌표와 방향별 px_per_deg를 사전 계산합니다."""
    cy, cx = h / 2.0, w / 2.0
    radius = int(min(h, w) * 0.45)
    t_vals = np.arange(-radius, radius + 1, 1.0)
    fov_w_deg = cpd_spec["fov_w_deg"]
    fov_h_deg = cpd_spec["fov_h_deg"]

    scan_lines = []
    for deg in range(0, 180, 15):
        rad = math.radians(deg)
        px_per_deg_dir = 1.0 / math.sqrt((math.cos(rad) / (w / fov_w_deg)) ** 2 + (math.sin(rad) / (h / fov_h_deg)) ** 2)
        for offset in np.linspace(-radius * 0.5, radius * 0.5, 5):
            xs = np.clip(np.round(cx + t_vals * math.cos(rad) - offset * math.sin(rad)).astype(int), 0, w - 1)
            ys = np.clip(np.round(cy + t_vals * math.sin(rad) + offset * math.cos(rad)).astype(int), 0, h - 1)
            scan_lines.append((ys, xs, px_per_deg_dir))
    return scan_lines

@torch.no_grad()
def evaluate_single_frame_all_metrics(rgb_255: torch.Tensor, scan_lines: list) -> dict:
    """
    단일 비디오 프레임(rgb_255)에서 5대 위험 트리거의 (위험 여부 bool, 방식 1 확률 float)을 한 번에 계산합니다.
    1) 휘도 플래시용 국소 고휘도 (면적 임계치 25%)
    2) 적색 플래시용 포화 적색 (면적 임계치 25%)
    3) 지속 플래시용 국소 고휘도 (면적 임계치 40%)
    4) 줄무늬 패턴 (면적 임계치 25%, 1.0~10.0 cpd 연속 5쌍 이상)
    5) 고대비 (면적 임계치 40%)
    """
    c_lin, _, lum = srgb_to_linear_and_luminance(rgb_255)
    h, w = lum.shape
    lum_4d = lum.unsqueeze(0).unsqueeze(0)

    # -------------------------------------------------------------------------
    # A. 국소 고휘도 평가 (25% -> 휘도 플래시용 / 40% -> 지속 플래시용)
    # -------------------------------------------------------------------------
    l_base = torch.clamp(lum.mean(), min=1.0)
    target_lum = torch.clamp(l_base * 1.30, min=20.0)
    high_lum_mask = lum >= target_lum
    high_lum_area = float(high_lum_mask.float().mean().item())

    rel_ratio_map = torch.clamp((lum - l_base) / torch.clamp(l_base * 0.30, min=1e-6), min=0.0)
    lum_gate = float(torch.clamp(rel_ratio_map[high_lum_mask].mean() if high_lum_mask.any() else rel_ratio_map.max(), 0.0, 1.0).item())

    # 1) 휘도 플래시용 (면적 임계치 25%)
    lum_flash_danger = high_lum_area > 0.25
    eta_lum_25 = (high_lum_area / 0.25) * lum_gate
    lum_flash_prob1 = (eta_lum_25 ** 2 / (1.0 + eta_lum_25 ** 2)) * 100.0

    # 3) 지속 플래시용 (면적 임계치 40%)
    sustained_danger = high_lum_area > 0.40
    eta_lum_40 = (high_lum_area / 0.40) * lum_gate
    sustained_prob1 = (eta_lum_40 ** 2 / (1.0 + eta_lum_40 ** 2)) * 100.0

    # -------------------------------------------------------------------------
    # B. 포화 적색 평가 (25% -> 적색 플래시용)
    # -------------------------------------------------------------------------
    r_lin, g_lin, b_lin = c_lin[..., 0], c_lin[..., 1], c_lin[..., 2]
    rgb_sum = r_lin + g_lin + b_lin
    red_ratio = torch.where(rgb_sum > 1e-4, r_lin / torch.clamp(rgb_sum, min=1e-6), torch.zeros_like(r_lin))
    sat_red_mask = (red_ratio >= 0.8) & (r_lin > 0.01)
    sat_red_area = float(sat_red_mask.float().mean().item())

    x_cie = 0.4124564 * r_lin + 0.3575761 * g_lin + 0.1804375 * b_lin
    y_cie = 0.2126729 * r_lin + 0.7151522 * g_lin + 0.0721750 * b_lin
    z_cie = 0.0193339 * r_lin + 0.1191920 * g_lin + 0.9503041 * b_lin
    denom = x_cie + 15.0 * y_cie + 3.0 * z_cie
    valid_chroma = denom > 1e-4

    u_prime = torch.where(valid_chroma, (4.0 * x_cie) / torch.clamp(denom, min=1e-6), torch.full_like(x_cie, 0.1978))
    v_prime = torch.where(valid_chroma, (9.0 * y_cie) / torch.clamp(denom, min=1e-6), torch.full_like(y_cie, 0.4683))

    non_red_mask = (~sat_red_mask) & valid_chroma
    if sat_red_mask.any() and non_red_mask.any():
        u_red_mean = u_prime[sat_red_mask].mean()
        v_red_mean = v_prime[sat_red_mask].mean()
        dist_map = torch.sqrt((u_prime[non_red_mask] - u_red_mean) ** 2 + (v_prime[non_red_mask] - v_red_mean) ** 2)
        delta_uv = float(torch.quantile(dist_map, 0.90).item())
    else:
        delta_uv = 0.0

    red_flash_danger = (sat_red_area > 0.25) and (delta_uv > 0.2)
    red_gate = float(torch.clamp(red_ratio[sat_red_mask].mean() / 0.8 if sat_red_mask.any() else red_ratio.max() / 0.8, 0.0, 1.0).item())
    chroma_gate = min(1.0, delta_uv / 0.2)
    eta_red_25 = (sat_red_area / 0.25) * red_gate * chroma_gate
    red_flash_prob1 = (eta_red_25 ** 2 / (1.0 + eta_red_25 ** 2)) * 100.0

    # -------------------------------------------------------------------------
    # C. 줄무늬 패턴 평가 (25%, 1.0 ~ 10.0 cpd 연속 5쌍 이상)
    # -------------------------------------------------------------------------
    k_sp = max(7, int(min(h, w) * 0.03) | 1)
    pad_sp = k_sp // 2
    l_high_sp = F.max_pool2d(lum_4d, kernel_size=k_sp, stride=1, padding=pad_sp).squeeze()
    l_low_sp = -F.max_pool2d(-lum_4d, kernel_size=k_sp, stride=1, padding=pad_sp).squeeze()
    delta_l_sp = l_high_sp - l_low_sp
    michelson_sp = delta_l_sp / torch.clamp(l_high_sp + l_low_sp, min=1e-6)
    contrast_ratio_sp = torch.where(l_low_sp < 160.0, delta_l_sp / 20.0, michelson_sp / (1.0 / 17.0))
    contrast_mask_sp = torch.where(l_low_sp < 160.0, delta_l_sp >= 20.0, michelson_sp > (1.0 / 17.0))

    env_k = max(15, int(min(h, w) * 0.06) | 1)
    pattern_density = F.avg_pool2d(contrast_mask_sp.float().unsqueeze(0).unsqueeze(0), kernel_size=env_k, stride=1, padding=env_k // 2).squeeze()
    pattern_region_mask = pattern_density >= 0.15
    pattern_area = float(pattern_region_mask.float().mean().item())

    lum_np = lum.detach().cpu().numpy()
    max_pairs = 0
    for ys, xs, px_per_deg_dir in scan_lines:
        pairs, _ = count_opposing_stripe_pairs_1d_cpd(
            lum_np[ys, xs],
            px_per_deg=px_per_deg_dir,
            cpd_min=1.0,
            cpd_max=10.0
        )
        if pairs > max_pairs:
            max_pairs = pairs

    stripe_danger = (max_pairs >= 5) and (pattern_area > 0.25)
    mean_contrast_factor = float(torch.clamp(contrast_ratio_sp[pattern_region_mask].mean() if pattern_region_mask.any() else contrast_ratio_sp.max(), 0.0, 1.0).item())
    pair_factor = min(1.0, max_pairs / 5.0)
    eta_stripe = (pattern_area / 0.25) * pair_factor * mean_contrast_factor
    stripe_prob1 = (eta_stripe ** 2 / (1.0 + eta_stripe ** 2)) * 100.0

    # -------------------------------------------------------------------------
    # D. 국소 고대비 영역 평가 (40%)
    # -------------------------------------------------------------------------
    k_hc = max(9, int(min(h, w) * 0.04) | 1)
    pad_hc = k_hc // 2
    l_high_hc = F.max_pool2d(lum_4d, kernel_size=k_hc, stride=1, padding=pad_hc).squeeze()
    l_low_hc = -F.max_pool2d(-lum_4d, kernel_size=k_hc, stride=1, padding=pad_hc).squeeze()
    delta_l_hc = l_high_hc - l_low_hc
    michelson_hc = delta_l_hc / torch.clamp(l_high_hc + l_low_hc, min=1e-6)

    hc_mask = torch.where(l_low_hc < 160.0, delta_l_hc >= 20.0, michelson_hc > (1.0 / 17.0))
    hc_severity = torch.where(l_low_hc < 160.0, delta_l_hc / 20.0, michelson_hc / (1.0 / 17.0))
    hc_area = float(hc_mask.float().mean().item())

    hc_danger = hc_area > 0.40
    hc_gate = float(torch.clamp(hc_severity[hc_mask].mean() if hc_mask.any() else hc_severity.max(), 0.0, 1.0).item())
    eta_hc_40 = (hc_area / 0.40) * hc_gate
    hc_prob1 = (eta_hc_40 ** 2 / (1.0 + eta_hc_40 ** 2)) * 100.0

    return {
        "lum_flash_danger": lum_flash_danger,
        "lum_flash_prob1": lum_flash_prob1,
        "red_flash_danger": red_flash_danger,
        "red_flash_prob1": red_flash_prob1,
        "sustained_danger": sustained_danger,
        "sustained_prob1": sustained_prob1,
        "stripe_danger": stripe_danger,
        "stripe_prob1": stripe_prob1,
        "hc_danger": hc_danger,
        "hc_prob1": hc_prob1,
    }

### [코드 셀 2] 5대 위험 트리거별 시계열 구간(점멸 · 지속 전환 · 패턴/고대비 통합) 탐지 함수

In [3]:
def _extract_danger_runs(is_danger_list: list[bool]) -> list[tuple[int, int]]:
    """불리언 리스트에서 연속된 True(위험 프레임) 묶음들의 (시작 프레임, 끝 프레임) 리스트를 반환합니다."""
    runs = []
    n = len(is_danger_list)
    i = 0
    while i < n:
        if is_danger_list[i]:
            start = i
            while i + 1 < n and is_danger_list[i + 1]:
                i += 1
            runs.append((start, i))
        i += 1
    return runs

def _merge_intervals(intervals: list[tuple[int, int]]) -> list[tuple[int, int]]:
    """겹치거나 인접한 (start_frame, end_frame) 구간들을 하나로 병합합니다."""
    if not intervals:
        return []
    sorted_iv = sorted(intervals, key=lambda x: (x[0], x[1]))
    merged = [sorted_iv[0]]
    for cur_s, cur_e in sorted_iv[1:]:
        prev_s, prev_e = merged[-1]
        if cur_s <= prev_e + 1:
            merged[-1] = (prev_s, max(prev_e, cur_e))
        else:
            merged.append((cur_s, cur_e))
    return merged

def detect_flashing_intervals(
    is_danger_list: list[bool],
    fps: float,
    min_pairs: int = 4,
    max_safe_gap_frames: int = 9
) -> list[tuple[int, int]]:
    """
    [방식 A - 점멸 탐지]
    연속된 위험 프레임 묶음(True)과 직후 이어지는 안전 프레임 묶음(False)을 1개의 '(위험, 안전) 쌍'으로 정의할 때,
    - 위험 프레임 묶음 사이의 안전 프레임 개수가 1 ~ max_safe_gap_frames(9프레임) 이하이고,
    - 1초(fps 프레임 길이 창) 안에 이러한 (위험, 안전) 쌍이 min_pairs(4번, > 3Hz) 이상 들어오는 구간을 검출합니다.
    """
    n_frames = len(is_danger_list)
    danger_runs = _extract_danger_runs(is_danger_list)
    if len(danger_runs) < min_pairs:
        return []

    # 유효한 (위험 묶음 k, 직후 안전 묶음 k) 쌍 생성
    # 쌍 k가 성립하려면 위험 묶음 k 뒤에 최소 1개 이상의 안전 프레임이 존재해야 함
    pairs = []
    for idx, (d_start, d_end) in enumerate(danger_runs):
        if d_end + 1 >= n_frames:
            # 영상의 마지막 프레임에서 끝나 뒤에 안전 프레임이 없는 경우 쌍 미성립
            continue
        next_d_start = danger_runs[idx + 1][0] if (idx + 1 < len(danger_runs)) else None
        gap_frames = (next_d_start - d_end - 1) if (next_d_start is not None) else (n_frames - 1 - d_end)
        pairs.append({
            "run_idx": idx,
            "d_start": d_start,
            "d_end": d_end,
            "first_safe_frame": d_end + 1,
            "gap_to_next_danger": gap_frames,
            "has_next_danger": next_d_start is not None
        })

    if len(pairs) < min_pairs:
        return []

    raw_intervals = []
    # 연속된 min_pairs(4개)의 쌍 (k, k+1, k+2, k+3) 검사
    for k in range(len(pairs) - min_pairs + 1):
        window_pairs = pairs[k : k + min_pairs]
        # 1) 연속된 위험 묶음들 사이의 안전 프레임 간격이 모두 9프레임 이하인지 확인
        gaps_ok = True
        for p_idx in range(min_pairs - 1):
            if (window_pairs[p_idx + 1]["run_idx"] != window_pairs[p_idx]["run_idx"] + 1) or (
                window_pairs[p_idx]["gap_to_next_danger"] > max_safe_gap_frames
            ):
                gaps_ok = False
                break
        if not gaps_ok:
            continue

        # 2) 첫 번째 쌍의 시작(d_start)부터 4번째 쌍이 완성되는 첫 안전 프레임(first_safe_frame)까지 1.0초 이내인지 확인
        seq_start_frame = window_pairs[0]["d_start"]
        seq_pair_complete_frame = window_pairs[-1]["first_safe_frame"]
        elapsed_sec = (seq_pair_complete_frame - seq_start_frame) / float(fps)

        if elapsed_sec <= 1.0:
            # 구간 범위는 첫 위험 프레임부터 마지막 쌍의 위험 프레임 끝(또는 안전 프레임)까지 기록
            raw_intervals.append((seq_start_frame, window_pairs[-1]["d_end"]))

    # 인접하거나 같은 점멸 체인(안전 간격 <= 9프레임)에 속한 플래시 구간들을 병합
    if not raw_intervals:
        return []
    sorted_iv = sorted(raw_intervals, key=lambda x: (x[0], x[1]))
    merged = [sorted_iv[0]]
    for cur_s, cur_e in sorted_iv[1:]:
        prev_s, prev_e = merged[-1]
        if cur_s <= prev_e + max_safe_gap_frames + 1:
            merged[-1] = (prev_s, max(prev_e, cur_e))
        else:
            merged.append((cur_s, cur_e))
    return merged

def detect_sustained_flash_intervals(
    is_danger_list: list[bool],
    fps: float,
    min_duration_sec: float = 1.0
) -> list[tuple[int, int]]:
    """
    [3. 지속 플래시 - 방식 A]
    국소 고휘도(40%) 안전 프레임(False)에서 위험 프레임(True)으로 전환(d_start >= 1)된 뒤,
    1.0초 이상 연속으로 위험 프레임이 지속되는 구간을 반환합니다.
    """
    danger_runs = _extract_danger_runs(is_danger_list)
    intervals = []
    for d_start, d_end in danger_runs:
        # 직전에 안전 프레임이 존재하여 '안전 -> 위험' 전환이 일어난 경우만 인정 (d_start >= 1)
        if d_start >= 1 and not is_danger_list[d_start - 1]:
            duration_sec = (d_end - d_start + 1) / float(fps)
            if duration_sec >= min_duration_sec:
                intervals.append((d_start, d_end))
    return _merge_intervals(intervals)

def detect_pattern_or_contrast_intervals(
    is_danger_list: list[bool],
    fps: float,
    min_duration_sec: float = 0.5,
    min_pairs: int = 4,
    max_safe_gap_frames: int = 9
) -> list[tuple[int, int]]:
    """
    [4. 줄무늬 패턴 (25%) & 5. 고대비 (40%) - 방식 A (논문 기준 통합)]
    1) 위험 프레임이 0.5초(Wilkins et al. 2005 / Ofcom 기준) 이상 연속 지속되는 구간, 또는
    2) (위험 프레임, 안전 프레임) 쌍이 1초에 4번 이상(위험 프레임 간 안전 프레임 9프레임 이하) 점멸하는 구간
    을 모두 탐지하여 병합 후 반환합니다.
    """
    # 조건 1: 0.5초 이상 연속 지속 구간
    danger_runs = _extract_danger_runs(is_danger_list)
    sustained_ivs = []
    for d_start, d_end in danger_runs:
        duration_sec = (d_end - d_start + 1) / float(fps)
        if duration_sec >= min_duration_sec:
            sustained_ivs.append((d_start, d_end))

    # 조건 2: 1초에 4쌍 이상 점멸 구간
    flash_ivs = detect_flashing_intervals(
        is_danger_list,
        fps=fps,
        min_pairs=min_pairs,
        max_safe_gap_frames=max_safe_gap_frames
    )

    return _merge_intervals(sustained_ivs + flash_ivs)

### [코드 셀 3] 동영상 전체 분석 실행 및 `위험 구간 요약 데이터프레임` 생성

In [4]:
@torch.no_grad()
def evaluate_video_pse_risk(
    video_path: str,
    device: str = "laptop",
    display_area: int = 15,
    max_side: int = MAX_SIDE
) -> pd.DataFrame:
    """
    동영상을 입력받아 5대 위험 트리거(휘도 플래시, 적색 플래시, 지속 플래시, 줄무늬 패턴, 고대비)에 따른
    위험 구간 요약 데이터프레임을 반환합니다.
    - 컬럼: ['위험 구분', '위험 구간 (프레임)', '위험 구간 (시간)', '판정 근거 - 임계치 달성 정도']
    """
    meta = get_video_metadata(video_path, max_side=max_side)
    proc_w, proc_h = meta["proc_w"], meta["proc_h"]
    fps = meta["fps"]

    cpd_spec = compute_viewing_angle_and_cpd_spec(
        proc_w, proc_h, device=device, display_area=display_area, cpd_min=1.0, cpd_max=10.0
    )
    scan_lines = precompute_stripe_scan_lines(proc_w, proc_h, cpd_spec)
    device_t = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 프레임별 5대 지표 기록 리스트
    history = {
        "lum_flash_danger": [], "lum_flash_prob1": [],
        "red_flash_danger": [], "red_flash_prob1": [],
        "sustained_danger": [], "sustained_prob1": [],
        "stripe_danger": [], "stripe_prob1": [],
        "hc_danger": [], "hc_prob1": [],
    }

    cap = cv2.VideoCapture(video_path)
    while True:
        ret, frame_bgr = cap.read()
        if not ret or frame_bgr is None:
            break
        img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        if img_rgb.shape[1] != proc_w or img_rgb.shape[0] != proc_h:
            img_rgb = cv2.resize(img_rgb, (proc_w, proc_h), interpolation=cv2.INTER_AREA)

        rgb_t = torch.from_numpy(img_rgb).to(device=device_t, dtype=torch.float32)
        res = evaluate_single_frame_all_metrics(rgb_t, scan_lines)
        for k in history:
            history[k].append(res[k])
    cap.release()

    # 5대 트리거별 시계열 위험 구간 검출
    trigger_Specs = [
        (
            "휘도 플래시",
            history["lum_flash_danger"],
            history["lum_flash_prob1"],
            detect_flashing_intervals(history["lum_flash_danger"], fps=fps, min_pairs=4, max_safe_gap_frames=9)
        ),
        (
            "적색 플래시",
            history["red_flash_danger"],
            history["red_flash_prob1"],
            detect_flashing_intervals(history["red_flash_danger"], fps=fps, min_pairs=4, max_safe_gap_frames=9)
        ),
        (
            "지속 플래시",
            history["sustained_danger"],
            history["sustained_prob1"],
            detect_sustained_flash_intervals(history["sustained_danger"], fps=fps, min_duration_sec=1.0)
        ),
        (
            "줄무늬 패턴",
            history["stripe_danger"],
            history["stripe_prob1"],
            detect_pattern_or_contrast_intervals(history["stripe_danger"], fps=fps, min_duration_sec=0.5, min_pairs=4, max_safe_gap_frames=9)
        ),
        (
            "고대비",
            history["hc_danger"],
            history["hc_prob1"],
            detect_pattern_or_contrast_intervals(history["hc_danger"], fps=fps, min_duration_sec=0.5, min_pairs=4, max_safe_gap_frames=9)
        ),
    ]

    rows = []
    for trigger_name, danger_list, prob1_list, intervals in trigger_Specs:
        for s_frame, e_frame in intervals:
            # 해당 구간 내 위험 프레임들의 방식 1 확률(임계치 달성도 기반) 평균 계산
            danger_probs = [
                prob1_list[f_idx]
                for f_idx in range(s_frame, e_frame + 1)
                if danger_list[f_idx]
            ]
            if not danger_probs:
                danger_probs = prob1_list[s_frame : e_frame + 1]
            mean_prob1 = float(np.mean(danger_probs)) if danger_probs else 0.0

            start_sec = s_frame / float(fps)
            end_sec = e_frame / float(fps)
            rows.append({
                "_sort_frame": s_frame,
                "위험 구분": trigger_name,
                "위험 구간 (프레임)": f"{s_frame} ~ {e_frame}",
                "위험 구간 (시간)": f"{start_sec:05.2f}s ~ {end_sec:05.2f}s",
                "판정 근거 - 임계치 달성 정도": f"{mean_prob1:.2f}%"
            })

    # 시작 프레임 순으로 정렬 후 내부 정렬 키 제거
    rows.sort(key=lambda r: (r["_sort_frame"], r["위험 구분"]))
    for r in rows:
        r.pop("_sort_frame", None)

    columns = ["위험 구분", "위험 구간 (프레임)", "위험 구간 (시간)", "판정 근거 - 임계치 달성 정도"]
    df_summary = pd.DataFrame(rows, columns=columns)
    return df_summary

df_video_result = evaluate_video_pse_risk(
    VIDEO_PATH,
    device=DEFAULT_DEVICE,
    display_area=DEFAULT_DISPLAY_AREA
)
display(df_video_result)

,위험 구분,위험 구간 (프레임),위험 구간 (시간),판정 근거 - 임계치 달성 정도
0,고대비,0 ~ 517,00.00s ~ 17.25s,86.21%
1,휘도 플래시,0 ~ 516,00.00s ~ 17.22s,86.16%
